---
title: "Revenue loss attribution"
author: "Andratx Bellmunt"
abstract: >
  Adapted from a real business scenario. We use mix-rate decomposition to asses the contribution of each individual product to a global loss of revenue per sale. In order to get bootstrap confidence intervals on each product contribution we rely on Monte Carlo methods. 
format:
  html:
    code-fold: true
    self-contained: true
    include-after-body: _tracker.html
jupyter: python3
number-sections: false
---

# Initialization

## Imports and settings

In [ ]:
import pandas as pd

In [ ]:
pd.options.display.float_format = "{:,.4f}".format

## Auxiliary functions

In [ ]:
def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    df["rps_before"] = df["revenue_before"] / df["sales_before"]
    df["rps_after"] = df["revenue_after"] / df["sales_after"]
    df["rps_diff"] = df["rps_after"] - df["rps_before"]

    return df

def aggregate_data(df: pd.DataFrame) -> pd.DataFrame:
    df_agg = pd.DataFrame(df.sum()).T
    df_agg["sales_before"] = df_agg["sales_before"].astype(int)
    df_agg["sales_after"] = df_agg["sales_after"].astype(int)

    return df_agg

def compute_rate_mix_effects(df: pd.DataFrame) -> pd.DataFrame:
    sales_total_after = df["sales_after"].sum()
    sales_total_before = df["sales_before"].sum()
    df["rate_effect"] = df["sales_after"] / sales_total_after * df["rps_diff"]
    df["mix_effect"] = (df["sales_after"] / sales_total_after - df["sales_before"] / sales_total_before) * df["rps_before"]
    df["total_effect"] = df["rate_effect"] + df["mix_effect"]

    return df


# Understanding the business problem

## The problem

In [ ]:
df_toy = pd.DataFrame(
    columns=["product", "sales_before", "revenue_before", "sales_after", "revenue_after"],
    data=[
        ["A", 200, 10000.00, 25, 1500.00],
        ["B",  40,   800.00, 40,  900.00]
    ]
)

df_toy = add_derived_columns(df_toy)
df_toy = compute_rate_mix_effects(df_toy)

df_agg = aggregate_data(df_toy)
df_agg = add_derived_columns(df_agg)

print("Toy example:")
display(df_toy)
print("Aggregated data:")
display(df_agg)

Even in a simple example with only two products we can see how the phenomenon of Simpson's paradox arises:

- Both products individual RPS goes up: +$10 for product A and +$2.50 for product B

- However the aggregated RPS goes down: -$8.08

This example actually illustrates a very common business situation:

- Product A runs under some algorithm that optimizes RPS

- The algorithm discards sales with low return and the final effect is that both revenue and sales decrease, albeit in a way that RPS increases (in other words: the percentual decrease in revenue is less than the percentual decrease in sales)

- When aggregating the data with other products this has a harming impact on the global RPS

In a real case scenario where we have ~4,000 different products instead of just a couple, these interactions become much more complex.

## The initial solution and why it doesn't work

In order to navigate the paradox we borrow a tool from game theory: [Shapley values](https://en.wikipedia.org/wiki/Shapley_value).

- Shapley values measure the contribution of each individual player to a common goal

- To us each product is a player and the common goal is the aggregated RPS. We want to measure how much each individual product contributes to it.

- One property of Shapley values is that individual contributions always add up to the final global result. In our example above, the sum of the Shapley value of A and the Shapley value of B must be -17.95.

*Note:* It is not our intend to provide a detailed account on how Shapley values are computed. In the present section we ask the reader to trust us, while in future more technical sections we assume the reader has enough familiarity with the concept.

In our example we have v(A) = $10.00, v(B) = $2.50, v(A,B) = -$8.08. Then:

- s(A) = 1/2 [v(A,B) + v(A) - v(B)] = -0.29

- s(B) = 1/2 [v(A,B) + v(B) - v(A)] = -7.79

In [ ]:
df_toy = compute_rate_mix_effects(df_toy)

In [ ]:
df_toy

In [ ]:
"""
Synthetic data generator for the rate-mix decomposition notebook.

Generates a realistic product catalog with before/after periods,
designed to produce a sharp aggregate RPS drop driven by a combination
of rate and mix effects, mimicking a real e-commerce scenario.

Product types
-------------
Type 1 — "Mix shifters" (~5% of products, ~40% of before-sales)
    Low-to-mid RPS products that experience a sharp volume surge.
    Their growing weight in the mix pulls the aggregate RPS down
    even though their individual RPS is stable. Pure mix effect.

Type 2 — "Rate decliners" (~15% of products, ~40% of before-sales)
    High RPS products that experience a sharp RPS drop with stable
    or slightly declining volume. Pure rate effect.

Type 3 — "Long tail" (~80% of products, ~20% of before-sales)
    Low-to-mid RPS, relatively stable in both dimensions.
    Background noise.

The combination of Type 1's volume surge at low RPS and Type 2's
rate drop produces the sharp aggregate RPS decline.
"""

import numpy as np
import pandas as pd
from scipy.stats import truncnorm


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _truncated_lognormal(mu, sigma, low, high, size, rng):
    samples = []
    while len(samples) < size:
        batch = rng.lognormal(mu, sigma, size=size * 3)
        batch = batch[(batch >= low) & (batch <= high)]
        samples.extend(batch.tolist())
    return np.array(samples[:size])


# ---------------------------------------------------------------------------
# Main generator
# ---------------------------------------------------------------------------

def generate_catalog(
    n_products: int = 4000,
    rps_low: float = 5.0,
    rps_high: float = 100.0,
    seed: int = 13,
) -> pd.DataFrame:
    """
    Generate a synthetic product catalog with before/after RPS and sales data.

    Parameters
    ----------
    n_products : int
        Total number of products.
    rps_low : float
        Minimum RPS in both periods.
    rps_high : float
        Maximum RPS in both periods.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    pd.DataFrame with columns:
        product_id, type,
        sales_before, revenue_before, rps_before,
        sales_after,  revenue_after,  rps_after,
        rps_change_pct, delta_rps_abs
    """
    rng = np.random.default_rng(seed)

    n_type1 = int(n_products * 0.05)
    n_type2 = int(n_products * 0.15)
    n_type3 = n_products - n_type1 - n_type2

    types = np.array(
        ["type1"] * n_type1 +
        ["type2"] * n_type2 +
        ["type3"] * n_type3
    )
    rng.shuffle(types)

    type1_mask = types == "type1"
    type2_mask = types == "type2"
    type3_mask = types == "type3"

    # --- Sales before -------------------------------------------------------
    # Overall lognormal base (80/20 rule)
    sales_before = _truncated_lognormal(
        mu=4.0, sigma=1.8, low=10, high=50_000, size=n_products, rng=rng
    ).astype(int)

    # Type 2 (rate decliners) are the high-volume, high-RPS products before
    sales_before[type2_mask] = (
        sales_before[type2_mask] *
        rng.uniform(4, 10, size=type2_mask.sum())
    ).astype(int)

    sales_before = np.maximum(sales_before, 1)

    # --- RPS before ---------------------------------------------------------
    rps_before = np.empty(n_products)

    # Type 1: low-to-mid RPS — these are cheap products that will surge
    rps_before[type1_mask] = _truncated_lognormal(
        mu=2.5, sigma=0.4, low=rps_low, high=25, size=type1_mask.sum(), rng=rng
    )
    # Type 2: high RPS — premium products that will deteriorate
    rps_before[type2_mask] = _truncated_lognormal(
        mu=4.2, sigma=0.3, low=60, high=rps_high, size=type2_mask.sum(), rng=rng
    )
    # Type 3: long tail, low-to-mid RPS
    rps_before[type3_mask] = _truncated_lognormal(
        mu=3.0, sigma=0.5, low=rps_low, high=50, size=type3_mask.sum(), rng=rng
    )
    rps_before = np.clip(rps_before, rps_low, rps_high)

    # --- Sales after --------------------------------------------------------
    sales_after = np.empty(n_products)

    # Type 1: sharp volume surge (+300% to +700%) — mix effect driver
    sales_after[type1_mask] = (
        sales_before[type1_mask] *
        rng.uniform(4.0, 8.0, size=type1_mask.sum())
    ).astype(int)

    # Type 2: mild volume decline (-10% to -30%)
    sales_after[type2_mask] = (
        sales_before[type2_mask] *
        rng.uniform(0.7, 0.9, size=type2_mask.sum())
    ).astype(int)

    # Type 3: stable with small noise (-15% to +15%)
    sales_after[type3_mask] = (
        sales_before[type3_mask] *
        rng.uniform(0.85, 1.15, size=type3_mask.sum())
    ).astype(int)

    sales_after = np.maximum(sales_after, 1).astype(int)

    # --- RPS after ----------------------------------------------------------
    rps_after = np.empty(n_products)

    # Type 1: stable RPS (-5% to +5%)
    rps_after[type1_mask] = rps_before[type1_mask] * rng.uniform(
        0.95, 1.05, size=type1_mask.sum()
    )
    # Type 2: sharp RPS drop (-65% to -80%) — rate effect driver
    rps_after[type2_mask] = rps_before[type2_mask] * rng.uniform(
        0.20, 0.35, size=type2_mask.sum()
    )
    # Type 3: mild drop, mostly stable (-10% to +5%)
    rps_after[type3_mask] = rps_before[type3_mask] * rng.uniform(
        0.90, 1.05, size=type3_mask.sum()
    )

    rps_after = np.clip(rps_after, rps_low, rps_high)

    # --- Revenue (derived) --------------------------------------------------
    revenue_before = (sales_before * rps_before).round(2)
    revenue_after  = (sales_after  * rps_after ).round(2)

    # --- Assemble -----------------------------------------------------------
    df = pd.DataFrame({
        "product_id":     np.arange(1, n_products + 1),
        "type":           types,
        "sales_before":   sales_before,
        "revenue_before": revenue_before,
        "rps_before":     rps_before.round(4),
        "sales_after":    sales_after,
        "revenue_after":  revenue_after,
        "rps_after":      rps_after.round(4),
    })

    df["rps_change_pct"] = (
        (df["rps_after"] / df["rps_before"] - 1) * 100
    ).round(4)
    df["delta_rps_abs"] = (df["rps_after"] - df["rps_before"]).round(4)

    return df


# ---------------------------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------------------------

def print_diagnostics(df: pd.DataFrame) -> None:

    total_sales_before = df["sales_before"].sum()
    total_sales_after  = df["sales_after"].sum()
    total_rev_before   = df["revenue_before"].sum()
    total_rev_after    = df["revenue_after"].sum()

    rps_global_before = total_rev_before / total_sales_before
    rps_global_after  = total_rev_after  / total_sales_after
    rps_global_change = (rps_global_after / rps_global_before - 1) * 100

    print("=" * 55)
    print("GLOBAL AGGREGATES")
    print("=" * 55)
    print(f"  Total sales before  : {total_sales_before:>12,.0f}")
    print(f"  Total sales after   : {total_sales_after:>12,.0f}")
    print(f"  Total revenue before: ${total_rev_before:>12,.0f}")
    print(f"  Total revenue after : ${total_rev_after:>12,.0f}")
    print(f"  Global RPS before   : ${rps_global_before:>10.4f}")
    print(f"  Global RPS after    : ${rps_global_after:>10.4f}")
    print(f"  Global RPS change   : {rps_global_change:>10.2f}%")

    print()
    print("=" * 55)
    print("BY PRODUCT TYPE")
    print("=" * 55)

    for t in ["type1", "type2", "type3"]:
        sub = df[df["type"] == t]
        n = len(sub)
        pct_products      = n / len(df) * 100
        pct_sales_before  = sub["sales_before"].sum() / total_sales_before * 100
        pct_sales_after   = sub["sales_after"].sum()  / total_sales_after  * 100
        median_rps_before = sub["rps_before"].median()
        median_rps_after  = sub["rps_after"].median()
        median_change     = sub["rps_change_pct"].median()

        print(f"\n  {t.upper()} ({n} products, {pct_products:.0f}% of catalog)")
        print(f"    Share of sales before : {pct_sales_before:.1f}%")
        print(f"    Share of sales after  : {pct_sales_after:.1f}%")
        print(f"    Median RPS before     : ${median_rps_before:.2f}")
        print(f"    Median RPS after      : ${median_rps_after:.2f}")
        print(f"    Median RPS change     : {median_change:.1f}%")

    print()
    print("=" * 55)
    print("SIMPSON'S PARADOX CHECK")
    print("=" * 55)
    n_improved = (df["rps_change_pct"] > 0).sum()
    n_declined = (df["rps_change_pct"] <= 0).sum()
    print(f"  Products with improved RPS : {n_improved} ({n_improved/len(df)*100:.1f}%)")
    print(f"  Products with declined RPS : {n_declined} ({n_declined/len(df)*100:.1f}%)")
    print(f"  Yet global RPS change      : {rps_global_change:.2f}%")
    print()




In [ ]:
df = generate_catalog(n_products=4000, seed=13)
print_diagnostics(df)

In [ ]:
df = df["product_id"]

In [ ]:
df_red = df[["product_id", "sales_before", "revenue_before", "sales_after", "revenue_after"]].copy()

In [ ]:
df_red = add_derived_columns(df_red)

In [ ]:
df_red = compute_rate_mix_effects(df_red)

In [ ]:
df_agg = aggregate_data(df_red)

In [ ]:
df_agg = add_derived_columns(df_agg)

In [ ]:
df_agg